# Kaleb Alejandro Aguilar Ávila
## Tarea 1 — Attention Recurrent Language Model and Machine Translation
**mLLMs / Procesamiento de Lenguaje Natural — CIMAT, Otoño 2026**  
**Profesor:** Dr. Adrián Pastor López Monroy

**LLM utilizado para Vibe Coding:** _(completar)_

---

### Índice

**0. Configuración**

**Parte 1 — Language Modelling (tuits MEX-A3T)**
- 1.0 Datos y vocabulario compartido
- 1.1 (Punto 1) Recurrent Language Model con Self-Attention (LSTM y GRU)
- 1.2 (Punto 2) Statistical Language Model
- 1.3 (Punto 3) Neural Language Model (Bengio)
- 1.4 (Punto 4) Ablación sin atención (LSTM y GRU) y tabla de perplejidades
- 1.5 (Punto 5) Generación de texto

**Parte 2 — Machine Translation inglés → español (Tatoeba)**
- 2.0 Datos
- 2.1 (Punto 1) Seq2Seq recurrente con atención Q, K, V
- 2.2 (Punto 2) Experimentos con X y 2X pares
- 2.3 (Punto 3) Evaluación en Test: BLEU, chrF y ejemplos

**3. Comentario general**

El código de los modelos vive en los módulos `.py` de `Task1_Language_Modelling/` y `Task2_Machine_Translation/`; este notebook sólo los importa y ejecuta los experimentos.

# 0. Configuración

In [ ]:
import os
import sys
import time
import random
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

T1_DIR = 'Task1_Language_Modelling'
T2_DIR = 'Task2_Machine_Translation'
sys.path.insert(0, os.path.abspath(T1_DIR))
sys.path.insert(0, os.path.abspath(T2_DIR))

# Parte 1
from common import (Vocab, load_tweets, train_dev_split, set_seed, get_device,
                    train_lm, perplexity, make_logger)
import slm
import nlm
import rnn_attention_lm as rlm

# Parte 2
import mt_data as D
import seq2seq_attention as S

SEED = 1111
set_seed(SEED)
device = get_device()
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

# Sólo se guarda el MEJOR checkpoint de cada modelo (se sobrescribe cuando mejora en dev/val)
CKPT_DIR = 'checkpoints'
# Cada entrenamiento escribe su log en logs/<modelo>.log  ->  en una terminal: tail -f logs/<modelo>.log
LOG_DIR = 'logs'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

---
# Parte 1 — Language Modelling

Datos: tuits MEX-A3T. **Train** (`mex20_train.txt`) para entrenar y **Val** (`mex20_val.txt`) para reportar la perplejidad.

**Modelos comparados (1 estadístico + 5 entrenamientos):**

| # | Modelo | Punto | Código |
|---|---|---|---|
| — | SLM de n-gramas (conteos, sin entrenamiento iterativo) | 2 | `Task1_Language_Modelling/slm.py` |
| 1 | LSTM + Self-Attention (Q, K, V) | 1 | `Task1_Language_Modelling/rnn_attention_lm.py` |
| 2 | GRU + Self-Attention (Q, K, V) | 1 | `Task1_Language_Modelling/rnn_attention_lm.py` |
| 3 | NLM de Bengio | 3 | `Task1_Language_Modelling/nlm.py` |
| 4 | LSTM sin atención (ablación) | 4 | `Task1_Language_Modelling/rnn_attention_lm.py` |
| 5 | GRU sin atención (ablación) | 4 | `Task1_Language_Modelling/rnn_attention_lm.py` |

## 1.0 Datos y vocabulario compartido

Para que las perplejidades sean comparables, **todos los modelos usan el mismo tokenizador (`TweetTokenizer`), el mismo vocabulario y la misma convención**: cada tuit es independiente, se predicen todas sus palabras más `</s>`, y las palabras fuera de vocabulario se mapean a `<unk>`.

$$PPL = \exp\Big(-\frac{1}{N}\sum_i \log P(w_i \mid \text{historia}_i)\Big)$$

Train se divide 90/10 en *train* / *dev*; *dev* sólo se usa para early stopping y para ajustar el SLM. **Val se usa únicamente para reportar la perplejidad final.**

In [ ]:
lm_train_full = load_tweets(os.path.join(T1_DIR, 'data', 'mex20_train.txt'))
lm_val = load_tweets(os.path.join(T1_DIR, 'data', 'mex20_val.txt'))
lm_train, lm_dev = train_dev_split(lm_train_full, dev_frac=0.1, seed=SEED)
print(f'train_full={len(lm_train_full)} | train={len(lm_train)} | dev={len(lm_dev)} | val={len(lm_val)}')

VOCAB_SIZE = 5000
vocab = Vocab(max_size=VOCAB_SIZE, min_freq=1, lower=True).fit(lm_train_full)
print('|V| =', len(vocab))
print(f'OOV rate val: {vocab.oov_rate(lm_val):.2%}')

lm_results = []   # aquí se acumulan las filas de la tabla del Punto 4
lm_train_full[:5]

### Configuración común de los modelos recurrentes

LSTM y GRU usan **exactamente la misma configuración**; sólo cambia `rnn_type`. Las variantes sin atención (Punto 4) también comparten todo, salvo `use_attention=False`.

In [ ]:
RNN_TYPES = ['lstm', 'gru']
BASE_CFG = dict(emb_dim=256, hidden_dim=512, num_layers=1, n_attn_layers=10, n_heads=16, dropout=0.1)
RNN_TRAIN_CFG = dict(lr=1e-3, num_epochs=300, patience=10, clip=1.0, log_every=20)

VARIANTES = {
    'att':    dict(use_attention=True),     # Punto 1
    'no_att': dict(use_attention=False),    # Punto 4: mismos bloques, sin atención
}

def rnn_label(rnn_type, variante):
    t, L = rnn_type.upper(), BASE_CFG['n_attn_layers']
    return {'att':    f'{t} + Self-Attention (QKV, {L} bloques)',
            'no_att': f'{t} sin atención ({L} bloques FF, ablación)'}[variante]

rnn_train_loader = rlm.make_loader(lm_train, vocab, batch_size=64, shuffle=True)
rnn_dev_loader   = rlm.make_loader(lm_dev,   vocab, batch_size=128)
rnn_val_loader   = rlm.make_loader(lm_val,   vocab, batch_size=128)

rnn_models, rnn_hists = {}, {}     # llaves: (rnn_type, variante)

def run_rnn(rnn_type, variante):
    # Entrena una variante, guarda sólo el mejor checkpoint + log y agrega su PPL a lm_results
    key, name = (rnn_type, variante), f'{rnn_type}_{variante}'
    cfg = dict(BASE_CFG, rnn_type=rnn_type, **VARIANTES[variante])
    set_seed(SEED)
    m = rlm.RNNAttentionLM(len(vocab), pad_id=vocab.pad_id, **cfg)
    n_params = rlm.count_parameters(m)
    print(f'=== {rnn_label(*key)} | parámetros: {n_params:,}')

    t0 = time.time()
    rnn_hists[key] = train_lm(m, rnn_train_loader, rnn_dev_loader, device, vocab.pad_id,
                              save_path=f'{CKPT_DIR}/{name}_best.pt',
                              log=make_logger(f'{LOG_DIR}/{name}.log'), **RNN_TRAIN_CFG)
    print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

    ppl = perplexity(m, rnn_val_loader, device, vocab.pad_id)
    print(f'{rnn_label(*key)} -> PPL val = {ppl:.2f}\n')
    lm_results.append({'Modelo': rnn_label(*key), 'Parámetros': n_params, 'PPL val': ppl})
    rnn_models[key] = m
    return m

## 1.1 (Punto 1) Recurrent Language Model con Self-Attention — LSTM y GRU

**Arquitectura** (`rnn_attention_lm.py`):
- Embedding (256) → **LSTM o GRU** (512, **unidireccional**, para no ver el futuro).
- **10 bloques apilados** tipo Transformer (`AttentionBlock`), cada uno:
  - **Masked Multi-Head Attention** (Vaswani et al., 2017), 16 cabezas, sobre la salida del bloque anterior `z`:
    - `Q = z W_Q`, `K = z W_K`, `V = z W_V`
    - `Attention(Q, K, V) = softmax(Q Kᵀ / √d_k + M) V`, con la máscara del decoder `M_ij = −∞` si `j > i`
    - `z ← LayerNorm(z + Dropout(Attention(z)))`
  - **Feed-forward** `Linear → ReLU → Linear`: `z ← LayerNorm(z + Dropout(FFN(z)))`
- Salida: dropout → `W_o` → softmax sobre el vocabulario.

**Entrenamiento:** Adam (lr = 1e-3), gradient clipping = 1.0, `ReduceLROnPlateau`, early stopping (patience 10) sobre *dev*. Sólo se guarda el mejor checkpoint de cada modelo.

In [ ]:
print(inspect.getsource(rlm.MaskedMultiHeadAttention))
print(inspect.getsource(rlm.AttentionBlock))

### Entrenamientos 1 y 2: LSTM + atención y GRU + atención
Logs: `logs/lstm_att.log`, `logs/gru_att.log`.

In [ ]:
for t in RNN_TYPES:
    run_rnn(t, 'att')

### Pesos de atención por bloque
La parte triangular superior siempre es 0 por la máscara del decoder (ninguna palabra atiende a las siguientes). Misma escala de color (0–1) en todos los bloques.

In [ ]:
frase = 'la verdad es que no me importa lo que digan'
L = BASE_CFG['n_attn_layers']
ncols = min(L, 5)
nrows = (L + ncols - 1) // ncols

for t in RNN_TYPES:
    model = rnn_models[(t, 'att')]
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    for l in range(nrows * ncols):
        ax = axes[l // ncols][l % ncols]
        if l >= L:
            ax.axis('off'); continue
        toks, A = rlm.attention_map(model, vocab, frase, device, layer=l)
        im = ax.imshow(A.numpy(), cmap='viridis', vmin=0, vmax=1)
        ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=90, fontsize=7)
        ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks, fontsize=7)
        ax.set_title(f'Bloque {l + 1}')
    fig.supxlabel('Key'); fig.supylabel('Query')
    fig.colorbar(im, ax=axes, shrink=0.6)
    fig.suptitle(f'{t.upper()} + Self-attention: pesos por bloque (promedio de {BASE_CFG["n_heads"]} cabezas)')
    plt.show()

**Observaciones (Punto 1):** _(completar: hiperparámetros elegidos y por qué, LSTM vs GRU, dificultades, alucinaciones del LLM, etc.)_

## 1.2 (Punto 2) Statistical Language Model

Modelo de n-gramas (`slm.py`) con:
- **Laplace / add-k:** $P(w\mid ctx) = \frac{C(ctx, w) + k}{C(ctx) + k|V|}$
- **Interpolación lineal (Jelinek-Mercer):** $P = \sum_{m=1}^{n} \lambda_m P_m(w \mid \text{últimas } m-1 \text{ palabras})$, con el unigrama suavizado add-k para que nunca sea 0.

El orden `n` y las lambdas se eligen con **dev** (el modelo se ajusta con *train*); el modelo final se re-ajusta con todo Train y se evalúa en **Val**.

In [ ]:
# Baseline: trigramas con Laplace (add-1)
slm_laplace = slm.NgramLM(vocab, n=3, smoothing='laplace', k=1.0).fit(lm_train_full)
ppl = slm_laplace.perplexity(lm_val)
print(f'SLM trigramas Laplace -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': 'SLM trigramas (Laplace)', 'PPL val': ppl})

In [ ]:
# Búsqueda de n y lambdas (unigrama, ..., n-grama) sobre dev
LAMBDA_GRID = {
    2: [[0.3, 0.7], [0.5, 0.5], [0.2, 0.8]],
    3: [[0.1, 0.3, 0.6], [0.2, 0.4, 0.4], [0.3, 0.4, 0.3], [0.2, 0.5, 0.3], [0.4, 0.4, 0.2]],
    4: [[0.1, 0.2, 0.3, 0.4], [0.1, 0.3, 0.3, 0.3], [0.2, 0.3, 0.3, 0.2], [0.2, 0.4, 0.3, 0.1]],
}
SLM_K = 0.1

slm_search = []
for n, grid in LAMBDA_GRID.items():
    for lambdas in grid:
        m = slm.NgramLM(vocab, n=n, smoothing='interpolation', k=SLM_K, lambdas=lambdas).fit(lm_train)
        slm_search.append({'n': n, 'lambdas': lambdas, 'PPL dev': m.perplexity(lm_dev)})

slm_search = pd.DataFrame(slm_search).sort_values('PPL dev').reset_index(drop=True)
best_n, best_lambdas = int(slm_search.loc[0, 'n']), list(slm_search.loc[0, 'lambdas'])
print(f'Mejor: n={best_n}, lambdas={best_lambdas}')
slm_search.round(2)

In [ ]:
slm_best = slm.NgramLM(vocab, n=best_n, smoothing='interpolation', k=SLM_K,
                       lambdas=best_lambdas).fit(lm_train_full)
ppl = slm_best.perplexity(lm_val)
print(f'SLM {best_n}-gramas interpolado -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': f'SLM {best_n}-gramas (interpolación {best_lambdas})', 'PPL val': ppl})

**Observaciones (Punto 2):** _(completar)_

## 1.3 (Punto 3) Neural Language Model de Bengio — entrenamiento 3

Ventana de 3 palabras → embeddings concatenados (100) → `tanh` (256) → dropout → softmax (`nlm.py`, basado en la Tarea 5 de NLP). Mismo esquema de entrenamiento y early stopping que los recurrentes. Log: `logs/bengio.log`.

In [ ]:
N = 4  # 3 palabras de contexto
nlm_train_loader = nlm.make_loader(lm_train, vocab, N, batch_size=256, shuffle=True)
nlm_dev_loader   = nlm.make_loader(lm_dev,   vocab, N, batch_size=512)
nlm_val_loader   = nlm.make_loader(lm_val,   vocab, N, batch_size=512)

set_seed(SEED)
bengio = nlm.BengioLM(len(vocab), N=N, emb_dim=100, hidden_dim=256, dropout=0.3, pad_id=vocab.pad_id)
print('Parámetros:', rlm.count_parameters(bengio))

t0 = time.time()
hist_nlm = train_lm(bengio, nlm_train_loader, nlm_dev_loader, device, vocab.pad_id,
                    save_path=f'{CKPT_DIR}/bengio_best.pt', log=make_logger(f'{LOG_DIR}/bengio.log'),
                    **dict(RNN_TRAIN_CFG, log_every=100))
print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

ppl = perplexity(bengio, nlm_val_loader, device, vocab.pad_id)
print(f'NLM Bengio -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': f'NLM Bengio (N={N})', 'Parámetros': rlm.count_parameters(bengio), 'PPL val': ppl})

**Observaciones (Punto 3):** _(completar)_

## 1.4 (Punto 4) Comparación de perplejidades

### Entrenamientos 4 y 5: ablación sin atención (LSTM y GRU)
`use_attention=False`: cada bloque conserva sólo la feed-forward (mismo número de bloques, residual y LayerNorm), sin la subcapa de atención; todo lo demás (datos, vocabulario, hiperparámetros, semilla) es idéntico. Logs: `logs/lstm_no_att.log`, `logs/gru_no_att.log`.

In [ ]:
for t in RNN_TYPES:
    run_rnn(t, 'no_att')

In [ ]:
fig, axes = plt.subplots(1, len(RNN_TYPES), figsize=(7 * len(RNN_TYPES), 4.5), sharey=True, squeeze=False)
for ax, t in zip(axes[0], RNN_TYPES):
    curvas = [('NLM Bengio', hist_nlm)] + [(rnn_label(t, v), rnn_hists[(t, v)]) for v in VARIANTES]
    for name, h in curvas:
        line, = ax.plot(range(1, len(h['dev_ppl']) + 1), h['dev_ppl'], marker='o', markersize=3, label=name)
        b = h['best_epoch']
        ax.scatter(b, h['dev_ppl'][b - 1], s=80, color=line.get_color(), edgecolor='black', zorder=3)
    ax.set_yscale('log'); ax.set_xlabel('Época'); ax.set_title(t.upper())
    ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=8)
axes[0][0].set_ylabel('PPL dev (escala log)')
fig.suptitle('Curvas de validación (dev) — el punto grande es la mejor época (checkpoint guardado)')
plt.tight_layout(); plt.show()

### Tabla de resultados (perplejidad sobre Val)

In [ ]:
lm_table = pd.DataFrame(lm_results).sort_values('PPL val').reset_index(drop=True)
lm_table['PPL val'] = lm_table['PPL val'].round(2)
lm_table['Parámetros'] = lm_table['Parámetros'].map(lambda p: f'{int(p):,}' if pd.notna(p) else '—')
lm_table

**Observaciones (Punto 4):** _(completar: ¿se derrotó al SLM y al NLM? LSTM vs GRU, ¿qué efecto tuvo quitar la atención?)_

## 1.5 (Punto 5) Generación de texto — 2 ejemplos por modelo

In [ ]:
prefixes = ['yo creo que', 'la verdad es']
TEMP = 0.8
set_seed(SEED)

for prefix in prefixes:
    print('=' * 100)
    print(f'Prefijo: "{prefix}"')
    print(f'  {"SLM":<45}:', slm_best.generate(prefix, max_len=25, temperature=TEMP, seed=SEED))
    print(f'  {"NLM Bengio":<45}:', nlm.generate(bengio, vocab, prefix, max_len=25, temperature=TEMP, device=device))
    for key, m in rnn_models.items():
        print(f'  {rnn_label(*key):<45}:', rlm.generate(m, vocab, prefix, max_len=25, temperature=TEMP, device=device))

**Demo: log-likelihood de oraciones** (una oración bien formada vs. una permutación; mayor = más probable)

In [ ]:
oraciones = ['yo creo que sí', 'sí que creo yo', 'no manches me cae mal']
rows = []
for s in oraciones:
    row = {'oración': s,
           'SLM': slm_best.sentence_logprob(s),
           'NLM Bengio': nlm.sentence_logprob(bengio, vocab, s, device)}
    for key, m in rnn_models.items():
        row[rnn_label(*key)] = rlm.sentence_logprob(m, vocab, s, device)
    rows.append(row)
pd.DataFrame(rows).set_index('oración').T.round(2)     # un modelo por fila

**Observaciones (Punto 5) y comentario de la Parte 1:** _(completar)_

---
# Parte 2 — Machine Translation (inglés → español)

Datos: Tatoeba (`tatoeba_train.tsv` 80 %, `tatoeba_val.tsv` 10 %, `tatoeba_test.tsv` 10 %).
- **Train** → entrenamiento (subconjuntos anidados de X y 2X pares).
- **Val** → early stopping / decisiones de hiperparámetros.
- **Test** → sólo evaluación final (BLEU y chrF). **Nunca** se usa para tomar decisiones.

Código: `Task2_Machine_Translation/mt_data.py` (datos, tokenización, vocabularios) y `Task2_Machine_Translation/seq2seq_attention.py` (modelo, entrenamiento, traducción y métricas).

## 2.0 Datos

In [ ]:
train_pairs = D.load_tatoeba(os.path.join(T2_DIR, 'data', 'tatoeba_train.tsv'))
val_pairs   = D.load_tatoeba(os.path.join(T2_DIR, 'data', 'tatoeba_val.tsv'))
test_pairs  = D.load_tatoeba(os.path.join(T2_DIR, 'data', 'tatoeba_test.tsv'))
print(f'|Train|={len(train_pairs)} | |Val|={len(val_pairs)} | |Test|={len(test_pairs)}')

lens_en = [len(D.tokenize(en)) for en, _ in train_pairs]
lens_es = [len(D.tokenize(es)) for _, es in train_pairs]
print(f'Longitud media (tokens) en={np.mean(lens_en):.1f} es={np.mean(lens_es):.1f} | '
      f'p99 en={np.percentile(lens_en, 99):.0f} es={np.percentile(lens_es, 99):.0f}')
train_pairs[:5]

## 2.1 (Punto 1) Seq2Seq recurrente con atención Q, K, V

**No es un Transformer:** encoder y decoder son recurrentes (GRU); sólo el bloque de atención es tipo Transformer.

- **Encoder:** Embedding (256) → GRU bidireccional de 2 capas (512) → salidas `H` (B, S, 1024).
- **Decoder:** GRU de 2 capas (512) con *input feeding* `[emb(y_{t-1}); z_{t-1}]`. En cada paso `t`:
  - **Query:** `Q = s_t · W_Q` ← estado oculto del decoder
  - **Keys:** `K = H · W_K` ← salidas del encoder
  - **Values:** `V = H · W_V` ← salidas del encoder
  - **Pesos de atención:** `α_t = softmax(Q Kᵀ / √d_k)` con máscara sobre `<pad>` (4 cabezas)
  - **Contexto:** `c_t = α_t · V` → `z_t = tanh(W_c [s_t; c_t])` → `softmax(W_o z_t)`
- **Entrenamiento:** teacher forcing, Adam (lr = 1e-3), label smoothing 0.1, gradient clipping 1.0, `ReduceLROnPlateau`, early stopping sobre la pérdida de Val. Decodificación greedy.

In [ ]:
print(inspect.getsource(S.QKVAttention))

## 2.2 (Punto 2) Experimentos con X y 2X pares

Ambos experimentos usan **exactamente la misma arquitectura y configuración de entrenamiento**; sólo cambia el número de pares de Train. Los X pares del experimento 1 son un **subconjunto** de los 2X del experimento 2 (Train se baraja una sola vez con semilla fija; se toman los primeros 2X y, de ésos, los primeros X).

Ajusta `X` según tus recursos de cómputo (debe cumplirse 2X ≤ |Train|). Los vocabularios se construyen sólo con los pares de entrenamiento de cada experimento.

In [ ]:
# GPU de 24 GB: la memoria no es el límite (el modelo usa pocos GB con batch 256); el límite es el tiempo.
X = 100_000   # <- 2X = 200,000 <= |Train| = 226,800. Bajar a 50_000 si el tiempo no alcanza.

MT_DATA_CFG  = dict(lower=True, min_freq=2, max_len_train=30, batch_size=256)
MT_MODEL_CFG = dict(emb_dim=256, hidden_dim=512, num_layers=2, attn_dim=256, n_heads=4, dropout=0.3)
MT_TRAIN_CFG = dict(lr=1e-3, num_epochs=20, patience=3, clip=1.0, teacher_forcing=1.0, label_smoothing=0.1,
                    log_every=100)

subset_x, subset_2x = D.make_nested_subsets(train_pairs, X, seed=SEED)
assert set(subset_x) <= set(subset_2x)
print(f'Experimento 1: {len(subset_x)} pares | Experimento 2: {len(subset_2x)} pares')

In [ ]:
def run_mt_experiment(name, pairs):
    """Construye vocabularios con `pairs`, entrena con early stopping sobre Val y regresa todo lo necesario."""
    set_seed(SEED)
    src_vocab, tgt_vocab = D.build_vocabs(pairs, min_freq=MT_DATA_CFG['min_freq'], lower=MT_DATA_CFG['lower'])
    print(f'[{name}] |V_en|={len(src_vocab)} |V_es|={len(tgt_vocab)}')

    train_loader = D.make_loader(pairs, src_vocab, tgt_vocab, MT_DATA_CFG['batch_size'], shuffle=True,
                                 lower=MT_DATA_CFG['lower'], max_len=MT_DATA_CFG['max_len_train'])
    val_loader = D.make_loader(val_pairs, src_vocab, tgt_vocab, MT_DATA_CFG['batch_size'],
                               lower=MT_DATA_CFG['lower'])

    model = S.Seq2SeqAttention(len(src_vocab), len(tgt_vocab), src_pad_id=src_vocab.pad_id,
                               tgt_pad_id=tgt_vocab.pad_id, **MT_MODEL_CFG).to(device)
    print(f'[{name}] parámetros: {S.count_parameters(model):,}')

    # BLEU sobre una muestra fija de Val para monitorear (NO Test)
    val_sample = val_pairs[:1000]
    def val_bleu():
        hyps = S.translate(model, [en for en, _ in val_sample], src_vocab, tgt_vocab, device,
                           batch_size=512, lower=MT_DATA_CFG['lower'])
        return S.compute_metrics(hyps, [es for _, es in val_sample], lowercase=MT_DATA_CFG['lower'])['BLEU']

    t0 = time.time()
    history = S.fit(model, train_loader, val_loader, tgt_vocab.pad_id, device,
                    save_path=f'{CKPT_DIR}/seq2seq_{name}_best.pt', val_bleu_fn=val_bleu,
                    log=S.make_logger(f'{LOG_DIR}/seq2seq_{name}.log'), **MT_TRAIN_CFG)
    train_time = time.time() - t0
    print(f'[{name}] tiempo de entrenamiento: {train_time / 60:.1f} min')
    return dict(model=model, src_vocab=src_vocab, tgt_vocab=tgt_vocab, history=history, train_time=train_time)

### Experimento 1 — X pares

In [ ]:
exp_x = run_mt_experiment('X', subset_x)

### Experimento 2 — 2X pares

In [ ]:
exp_2x = run_mt_experiment('2X', subset_2x)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, exp in [('X', exp_x), ('2X', exp_2x)]:
    h = exp['history']
    ep = range(1, len(h['val_loss']) + 1)
    ax[0].plot(ep, h['train_loss'], '--', label=f'train {name}')
    ax[0].plot(ep, h['val_loss'], marker='o', label=f'val {name}')
    ax[1].plot(ep, h['val_bleu'], marker='o', label=name)
ax[0].set_title('Loss'); ax[0].set_xlabel('Época'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('BLEU (muestra de Val)'); ax[1].set_xlabel('Época'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**Observaciones (Punto 2):** _(completar: valor de X elegido y por qué, recursos de cómputo, tiempo de entrenamiento)_

## 2.3 (Punto 3) Evaluación en Test — BLEU y chrF

In [ ]:
test_src = [en for en, _ in test_pairs]
test_ref = [es for _, es in test_pairs]

mt_rows = []
for name, exp, n in [('X', exp_x, X), ('2X', exp_2x, 2 * X)]:
    t0 = time.time()
    exp['test_hyps'] = S.translate(exp['model'], test_src, exp['src_vocab'], exp['tgt_vocab'], device,
                                   batch_size=512, lower=MT_DATA_CFG['lower'])
    m = S.compute_metrics(exp['test_hyps'], test_ref, lowercase=MT_DATA_CFG['lower'])
    print(name, m['bleu_obj'], '|', m['chrf_obj'], f'| {time.time() - t0:.0f}s')
    mt_rows.append({'Experimento': name, 'Pares de Train': n, 'Épocas': len(exp['history']['val_loss']), 'Mejor época': exp['history']['best_epoch'],
                    'Val loss': round(exp['history']['best_val_loss'], 3),
                    'BLEU (Test)': round(m['BLEU'], 2), 'chrF (Test)': round(m['chrF'], 2),
                    'Tiempo (min)': round(exp['train_time'] / 60, 1)})

pd.DataFrame(mt_rows)

### Ejemplos de Test (≥ 10)

In [ ]:
rng = random.Random(SEED)
idx = rng.sample(range(len(test_pairs)), 12)
examples = pd.DataFrame({
    'Inglés (original)': [test_src[i] for i in idx],
    'Referencia (es)': [test_ref[i] for i in idx],
    'Modelo X': [exp_x['test_hyps'][i] for i in idx],
    'Modelo 2X': [exp_2x['test_hyps'][i] for i in idx],
})
with pd.option_context('display.max_colwidth', None):
    display(examples)

### Mapa de atención (ejemplo)

In [ ]:
sentence = test_src[idx[0]]
src_toks, out_toks, A = S.translate_with_attention(exp_2x['model'], sentence, exp_2x['src_vocab'],
                                                   exp_2x['tgt_vocab'], device)
plt.figure(figsize=(6, 5))
plt.imshow(A.numpy(), cmap='viridis', aspect='auto')
plt.xticks(range(len(src_toks)), src_toks, rotation=90); plt.yticks(range(len(out_toks)), out_toks)
plt.xlabel('Inglés (Keys/Values)'); plt.ylabel('Español generado (Queries)'); plt.colorbar()
plt.title('Pesos de atención (promedio de cabezas) — modelo 2X'); plt.show()

**Observaciones (Punto 3):** _(completar: comparación X vs 2X, errores y diferencias observadas en los ejemplos)_

---
# 3. Comentario general

_(completar: LLM usado para Vibe Coding, decisiones tomadas, dificultades, alucinaciones, recursos de cómputo y tiempos de entrenamiento)_